In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import pymc.sampling_jax
import bambi as bmb
import scipy.stats as stats
from scipy.stats import gaussian_kde
from sklearn.preprocessing import scale
import arviz as az
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
%config InlineBackend.figure_format = 'retina'
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
az.style.use("arviz-darkgrid")
sns.set_theme(palette="colorblind")

In [3]:
# Initialize a random number generator for reproducibility
rng = np.random.default_rng(seed=42)

# Generate date range
dates = pd.date_range(start="2020-05-01", end="2020-07-01")
n_dates = len(dates)

# Cities and their average temperatures
cities = ["Berlin", "Paris", "Rome"]
avg_temps = [15, 17, 20]
n_cities = len(cities)

# Pre-allocate a NumPy array to hold the temperature data
temperature_array = np.zeros(n_dates * n_cities)

# Populate the temperature array
for i, avg_temp in enumerate(avg_temps):
    temperature_array[i * n_dates : (i + 1) * n_dates] = rng.normal(
        loc=avg_temp, scale=2, size=n_dates
    )

# Create date and city columns, repeating as necessary
date_column = np.tile(dates, n_cities)
city_column = np.repeat(cities, n_dates)

# Create a DataFrame in long format
df_data_long = pd.DataFrame(
    {"date": date_column, "city": city_column, "temperature": temperature_array}
)

# Show first few rows
print(df_data_long.head())

        date    city  temperature
0 2020-05-01  Berlin    15.609434
1 2020-05-02  Berlin    12.920032
2 2020-05-03  Berlin    16.500902
3 2020-05-04  Berlin    16.881129
4 2020-05-05  Berlin    11.097930


In [4]:
# The "coordinates" are the unique values that these dimensions can take
coords = {
    "date": pd.Categorical(df_data_long["date"]).categories,
    "city": pd.Categorical(df_data_long["city"]).categories,
}
print(coords)

{'date': DatetimeIndex(['2020-05-01', '2020-05-02', '2020-05-03', '2020-05-04',
               '2020-05-05', '2020-05-06', '2020-05-07', '2020-05-08',
               '2020-05-09', '2020-05-10', '2020-05-11', '2020-05-12',
               '2020-05-13', '2020-05-14', '2020-05-15', '2020-05-16',
               '2020-05-17', '2020-05-18', '2020-05-19', '2020-05-20',
               '2020-05-21', '2020-05-22', '2020-05-23', '2020-05-24',
               '2020-05-25', '2020-05-26', '2020-05-27', '2020-05-28',
               '2020-05-29', '2020-05-30', '2020-05-31', '2020-06-01',
               '2020-06-02', '2020-06-03', '2020-06-04', '2020-06-05',
               '2020-06-06', '2020-06-07', '2020-06-08', '2020-06-09',
               '2020-06-10', '2020-06-11', '2020-06-12', '2020-06-13',
               '2020-06-14', '2020-06-15', '2020-06-16', '2020-06-17',
               '2020-06-18', '2020-06-19', '2020-06-20', '2020-06-21',
               '2020-06-22', '2020-06-23', '2020-06-24', '2020-06-25

In [5]:
RANDOM_SEED = 123  

with pm.Model(coords=coords) as model:
    # Constant Data
    data = pm.Data("observed_temp", df_data_long["temperature"], dims=("obs_id"))
    
    # Coordinates for the observed data
    date_idx = pd.Categorical(df_data_long["date"]).codes
    city_idx = pd.Categorical(df_data_long["city"]).codes
    
    # Priors
    europe_mean = pm.Normal("europe_mean_temp", mu=15.0, sigma=3.0)
    city_offset = pm.Normal("city_offset", mu=0.0, sigma=3.0, dims="city")
    
    # Expected city temperature
    city_temperature = pm.Deterministic(
        "expected_city_temp", europe_mean + city_offset[city_idx], dims="obs_id"
    )
    
    # Model Error
    sigma = pm.Exponential("sigma", 1)
    
    # Likelihood
    pm.Normal("temperature", mu=city_temperature, sigma=sigma, observed=data, dims=("obs_id"))
    
    # Sampling
    idata = pm.sampling_jax.sample_numpyro_nuts()


/Users/corrado/opt/anaconda3/envs/pymc_env/lib/python3.11/site-packages/pymc/data.py:433: UserWarning: The `mutable` kwarg was not specified. Before v4.1.0 it defaulted to `pm.Data(mutable=True)`, which is equivalent to using `pm.MutableData()`. In v4.1.0 the default changed to `pm.Data(mutable=False)`, equivalent to `pm.ConstantData`. Use `pm.ConstantData`/`pm.MutableData` or pass `pm.Data(..., mutable=False/True)` to avoid this warning.
  warnings.warn(
Compiling...


Compilation time = 0:00:02.517465


Sampling...


  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

Sampling time = 0:00:03.036068


Transforming variables...


Transformation time = 0:00:00.206260


In [6]:
az.summary(idata)

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
europe_mean_temp,16.734,1.500,14.007,19.683,0.050,0.035,898.0,1220.0,1.01
city_offset[Berlin],-1.659,1.510,-4.673,1.011,0.050,0.036,898.0,1130.0,1.01
city_offset[Paris],-0.013,1.507,-2.958,2.746,0.050,0.039,909.0,1231.0,1.01
city_offset[Rome],3.265,1.507,0.261,5.968,0.050,0.036,893.0,1268.0,1.01
sigma,1.752,0.094,1.580,1.933,0.003,0.002,1171.0,1263.0,1.00
...,...,...,...,...,...,...,...,...,...
expected_city_temp[181],19.999,0.222,19.585,20.424,0.003,0.002,4357.0,3124.0,1.00
expected_city_temp[182],19.999,0.222,19.585,20.424,0.003,0.002,4357.0,3124.0,1.00
expected_city_temp[183],19.999,0.222,19.585,20.424,0.003,0.002,4357.0,3124.0,1.00
expected_city_temp[184],19.999,0.222,19.585,20.424,0.003,0.002,4357.0,3124.0,1.00
